# Observation-Level Feature Contribution

## Goal

This notebook teaches how a model uses features for **one specific observation**.

We will answer two questions:

1. In a **linear model**, how much does each feature contribute through `weight x value`?
2. In a **decision tree**, which feature-based questions route the observation to its final prediction?

We use the same dataset for both models so students can compare the explanation styles directly.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

np.random.seed(7)
print('Libraries loaded.')

## Part 1 - Build a Small Teaching Dataset

We want to predict `final_score` for students from a few measurable features:

- `study_hours`
- `attendance`
- `practice_questions`
- `previous_score`

The target is continuous, so we use **regression** models. This keeps the arithmetic easy to inspect.

In [ ]:
n = 18

study_hours = np.array([2, 3, 4, 5, 6, 7, 8, 9, 3, 4, 5, 6, 7, 8, 2, 9, 10, 6], dtype=float)
attendance = np.array([55, 60, 62, 68, 72, 75, 80, 88, 58, 64, 70, 73, 78, 84, 50, 90, 95, 76], dtype=float)
practice_questions = np.array([8, 12, 18, 22, 28, 35, 40, 48, 10, 16, 21, 30, 36, 42, 6, 50, 55, 33], dtype=float)
previous_score = np.array([42, 48, 52, 56, 60, 64, 68, 74, 46, 50, 58, 61, 66, 70, 40, 78, 82, 63], dtype=float)

final_score = (
    12
    + 1.8 * study_hours
    + 0.25 * attendance
    + 0.12 * practice_questions
    + 0.42 * previous_score
    + np.array([-2, 1, 0, -1, 2, 1, 0, 2, -1, 1, 0, 1, 2, 0, -2, 1, 2, 0], dtype=float)
)

df = pd.DataFrame({
    'study_hours': study_hours,
    'attendance': attendance,
    'practice_questions': practice_questions,
    'previous_score': previous_score,
    'final_score': final_score,
})

df

## Part 2 - Train Two Models

We train:

1. `LinearRegression`
2. `DecisionTreeRegressor`

Both models will try to predict the same target from the same features.

In [ ]:
feature_cols = ['study_hours', 'attendance', 'practice_questions', 'previous_score']
X = df[feature_cols]
y = df['final_score']

linear_model = LinearRegression()
linear_model.fit(X, y)

tree_model = DecisionTreeRegressor(max_depth=3, random_state=7)
tree_model.fit(X, y)

print('Linear model intercept:', round(linear_model.intercept_, 3))
print('Linear model weights:')
print(pd.Series(linear_model.coef_, index=feature_cols).round(3))

## Part 3 - Feature Contribution in a Linear Model

A linear regression prediction has the form:

$$\hat{y} = b + w_1x_1 + w_2x_2 + \dots + w_nx_n$$

For **one observation**, each feature contributes:

$$\text{feature contribution} = w_i x_i$$

The final prediction is:

- intercept
- plus contribution from feature 1
- plus contribution from feature 2
- and so on

In [ ]:
row_index = 5
observation = X.iloc[row_index]
actual_value = y.iloc[row_index]

observation

In [ ]:
weights = pd.Series(linear_model.coef_, index=feature_cols)
contributions = weights * observation

linear_breakdown = pd.DataFrame({
    'feature_value': observation,
    'weight': weights,
    'weight_x_value': contributions
}).round(3)

linear_prediction = linear_model.predict(observation.to_frame().T)[0]

print('Selected row index:', row_index)
print('Actual final_score:', round(actual_value, 3))
print('Linear prediction:', round(linear_prediction, 3))
print('Intercept contribution:', round(linear_model.intercept_, 3))
print()
linear_breakdown

In [ ]:
reconstructed_prediction = linear_model.intercept_ + contributions.sum()
print('Intercept + sum(weight x value) =', round(reconstructed_prediction, 3))

In [ ]:
linear_breakdown['weight_x_value'].sort_values().plot(kind='barh', color='#2a6f97', figsize=(8, 4))
plt.title('Linear model contribution for one observation')
plt.xlabel('Contribution to prediction')
plt.ylabel('Feature')
plt.show()

### Interpretation

In the linear model:

- every feature contributes numerically
- the contribution is always `weight x value`
- all contributions are added together with the intercept

So the model behaves like a weighted voting system.

## Part 4 - Feature Contribution in a Decision Tree

A decision tree works differently.

It does **not** compute one additive term per feature. Instead, it asks a sequence of questions such as:

- Is `previous_score <= 62.5`?
- Is `attendance <= 74.0`?

For one observation, the important explanation is:

- which features were used on the path
- which branch the observation took at each split
- which leaf produced the final prediction

In [ ]:
tree_prediction = tree_model.predict(observation.to_frame().T)[0]
leaf_id = tree_model.apply(observation.to_frame().T)[0]

print('Decision tree prediction:', round(tree_prediction, 3))
print('Leaf node reached:', leaf_id)

In [ ]:
node_indicator = tree_model.decision_path(observation.to_frame().T)
node_index = node_indicator.indices[node_indicator.indptr[0]:node_indicator.indptr[1]]

tree_ = tree_model.tree_
path_rows = []

for node_id in node_index:
    if tree_.feature[node_id] != -2:
        feature_name = feature_cols[tree_.feature[node_id]]
        threshold = tree_.threshold[node_id]
        value = observation[feature_name]
        decision = 'left' if value <= threshold else 'right'
        rule = f'{feature_name} <= {threshold:.3f}'
        path_rows.append({
            'node': node_id,
            'rule': rule,
            'observation_value': round(float(value), 3),
            'branch_taken': decision
        })
    else:
        path_rows.append({
            'node': node_id,
            'rule': 'leaf',
            'observation_value': np.nan,
            'branch_taken': f'predict {tree_.value[node_id][0][0]:.3f}'
        })

path_df = pd.DataFrame(path_rows)
path_df

In [ ]:
plt.figure(figsize=(16, 8))
plot_tree(
    tree_model,
    feature_names=feature_cols,
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title('Decision tree used for the lesson')
plt.show()

### Interpretation

In the decision tree:

- only the features on the selected path matter for this observation
- a feature contributes by sending the observation left or right at a split
- the leaf node stores the final prediction

So the tree behaves like a sequence of conditional rules rather than an additive weighted sum.

## Part 5 - Compare the Two Explanation Styles

| Model | How one feature contributes | Prediction style |
|---|---|---|
| Linear model | `weight x value` | Add all contributions |
| Decision tree | Split decision on a path | Follow rules to a leaf |

This difference is central in machine learning:

- linear models explain predictions through **parameters**
- decision trees explain predictions through **rules and branches**

## Key Takeaways

1. For a linear model, every feature usually contributes to every prediction.
2. The contribution for one row is `weight x feature value`.
3. For a decision tree, only features used on the path contribute for that row.
4. A tree explanation is about routing decisions, not additive weights.
5. The same dataset can be explained in very different ways depending on the model family.

### Suggested student exercise

Change `row_index` to another student and compare:

- which linear contributions increase or decrease
- whether the tree follows a different path
- whether the two models produce similar predictions